# task22: Can a team’s and its opponent’s recent records predict the team’s goals?

The analysis uses eight pre-match predictors. Targets include extra time and exclude shootout kicks.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'analysis.py').exists(): ROOT=ROOT/'Assignment3'
assert (ROOT/'analysis.py').exists()
sys.path.insert(0,str(ROOT.resolve()))
from analysis import prepare_data, run_task, TASK1_X, TASK2_X

one,two=prepare_data()
df=two
features=TASK2_X
print(features)
print(df.groupby('split').size().to_string())

['team_pre_gf', 'team_pre_ga', 'team_pre_win_rate', 'team_pre_clean_sheet_rate', 'opponent_pre_gf', 'opponent_pre_ga', 'opponent_pre_win_rate', 'opponent_pre_clean_sheet_rate']
split
test           44
train         112
validation     52


## T1–T3: baseline, multiple regression and exploration

Fit ordinary least squares (OLS) only on training rows. Compare classroom transformations and a training-mean baseline using identical splits. All primary regressions retain eight X variables. MAE is average absolute error; MSE averages squared errors; RMSE is its square root in goals. NRMSE divides RMSE by the training target range (a training-only adaptation of the classroom denominator). R² compares errors with predicting the evaluation-set mean; it can be negative. No predictor imputation is needed.

In [2]:
summary=run_task(df,features,'goals','task22')
out=ROOT/'results/task22'
print(pd.read_csv(out/'training_descriptive_statistics.csv').to_string(index=False))

Unnamed: 0  team_pre_gf  team_pre_ga  team_pre_win_rate  team_pre_clean_sheet_rate  opponent_pre_gf  opponent_pre_ga  opponent_pre_win_rate  opponent_pre_clean_sheet_rate      goals
     count   112.000000   112.000000         112.000000                 112.000000       112.000000       112.000000             112.000000                     112.000000 112.000000
      mean     1.714286     1.142857           0.464286                   0.341071         1.714286         1.142857               0.464286                       0.341071   1.482143
       std     0.771900     0.664372           0.268750                   0.236549         0.771900         0.664372               0.268750                       0.236549   1.470320
       min     0.200000     0.000000           0.000000                   0.000000         0.200000         0.000000               0.000000                       0.000000   0.000000
       25%     1.200000     0.600000           0.200000                   0.200000        

![Training scatterplots](results/task22/training_scatterplots.png)

Relationships are not causal effects.

## Simple versus multiple linear regression

This supplementary comparison uses one raw predictor versus all eight raw predictors. The single predictor is previous-five-match mean goals scored: `team_pre_gf` for Task 2.2 (the meaning of `team_previous_mean_goals_for`), or `a_pre_gf` for Team A in Task 2.1. It is fixed by the research question, not chosen by test performance. Both models use OLS with an intercept, the same training rows and exactly the same test rows. No transformations are used in this comparison.

MAE and RMSE are in goals; NRMSE divides RMSE by the same training-response range. Lower values indicate smaller prediction errors. Both models belong to the same algorithm family. This comparison does not establish compliance with a requirement for different algorithms, and the one-predictor model does not replace the required eight-predictor analysis. Evaluation is retrospective because these test outcomes have previously been inspected.

In [3]:
comparison=pd.read_csv(out/'simple_vs_multiple_metrics.csv')
print(comparison[['model','predictor_count','MAE','RMSE','NRMSE']].to_string(index=False))
print((out/'simple_vs_multiple_details.json').read_text())
simple_rmse,multiple_rmse=comparison.RMSE
print(f'RMSE reduction from adding seven predictors: {simple_rmse-multiple_rmse:.6f} goals')
print('The eight-predictor model has lower test RMSE.' if multiple_rmse < simple_rmse else 'The eight-predictor model does not have lower test RMSE.')
print('This describes this reused test period; it does not guarantee future performance.')

           model  predictor_count      MAE     RMSE    NRMSE
  simple_raw_ols                1 1.025496 1.290946 0.184421
multiple_raw_ols                8 1.011375 1.327271 0.189610
{
  "simple_feature": "team_pre_gf",
  "feature_meaning": "Mean goals scored in the previous five completed internationals",
  "simple_intercept": 0.8696863659927428,
  "simple_coefficient": 0.3572662865042334,
  "multiple_features": [
    "team_pre_gf",
    "team_pre_ga",
    "team_pre_win_rate",
    "team_pre_clean_sheet_rate",
    "opponent_pre_gf",
    "opponent_pre_ga",
    "opponent_pre_win_rate",
    "opponent_pre_clean_sheet_rate"
  ],
  "multiple_intercept": 2.2172482709484296,
  "multiple_coefficients": [
    -0.006488813518781996,
    -0.19587240322169053,
    1.4219346448673622,
    -0.7520017521680781,
    -0.1314845985634186,
    0.051512237228834634,
    -1.60353662653459,
    0.021124607334042034
  ],
  "training_rows": 112,
  "test_rows": 44,
  "nrmse_denominator": 7.0,
  "note": "Same tra

## T4–T6: log transformation

Both tasks now use nonnegative team records. Use log(1 + X) to handle zeros; this is a stated adaptation of the classroom log transform. Leave the response untransformed, including signed goal differences. Predictor skewness is not automatically a regression-assumption violation.

## T7–T8: correlation and reduction

![Correlations](results/task22/training_correlation.png)

The teaching exercise screens training correlations above |r| = 0.8 and retains the earlier-listed column. This is not a full diagnosis. Reduced models do not replace the required eight-predictor primary models.

In [4]:
print((out/'t8_reduction_exercise.json').read_text())

{
  "retained_features": [
    "team_pre_gf",
    "team_pre_ga",
    "team_pre_win_rate",
    "team_pre_clean_sheet_rate",
    "opponent_pre_gf",
    "opponent_pre_ga",
    "opponent_pre_win_rate",
    "opponent_pre_clean_sheet_rate"
  ],
  "validation": {
    "MAE": 0.8764672524226005,
    "MSE": 1.2907486486263187,
    "RMSE": 1.1361111955377954,
    "NRMSE": 0.1623015993625422,
    "R2": 0.20875439902843662
  },
  "note": "Classroom exercise only. Not eligible for final selection because the brief requires eight predictors."
}


## T9–T10: standardisation and Yeo–Johnson

Fit transformation parameters on training X only. Scaling OLS should not change fitted predictions apart from numerical precision. Log1p and Yeo–Johnson are transformations, not different regression algorithms. At the group’s request, no Ridge or other advanced algorithm is used. Full satisfaction of the HD competing-algorithms criterion is not claimed.

In [5]:
print(pd.read_csv(out/'validation_comparison.csv').to_string(index=False))
print('Best regression:',summary['selected_model'])
print('Validation recommendation including baseline:',summary['validation_recommendation'])

                 model      MAE      MSE     RMSE    NRMSE        R2
               raw_ols 0.876467 1.290749 1.136111 0.162302  0.208754
      standardised_ols 0.876467 1.290749 1.136111 0.162302  0.208754
        yeojohnson_ols 0.864677 1.277479 1.130256 0.161465  0.216889
             log1p_ols 0.861947 1.264698 1.124588 0.160655  0.224724
training_mean_baseline 1.053571 1.632874 1.277840 0.182549 -0.000973
Best regression: log1p_ols
Validation recommendation including baseline: log1p_ols


![Validation comparison](results/task22/validation_comparison.png)

## Equation and training summary

Coefficients multiply the selected transformed X values. Statsmodels verifies the same OLS model, not a competing algorithm. Training adjusted R² differs from evaluation R². Usual standard errors/p-values may be unreliable with repeated teams and paired rows; no strong independent-sample significance claims are made.

In [6]:
print((out/'selected_equation.txt').read_text())
print((out/'statsmodels_training_summary.txt').read_text())

Predicted goals = 2.619247 +0.159669 * team_pre_gf -0.929506 * team_pre_ga +1.414958 * team_pre_win_rate -1.556030 * team_pre_clean_sheet_rate -0.703824 * opponent_pre_gf +0.589966 * opponent_pre_ga -1.559976 * opponent_pre_win_rate +0.407149 * opponent_pre_clean_sheet_rate
Features in this equation are log(1 + raw historical feature).
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.197
Model:                            OLS   Adj. R-squared:                  0.134
Method:                 Least Squares   F-statistic:                     3.151
Date:                Tue, 13 Oct 2026   Prob (F-statistic):            0.00313
Time:                        15:15:38   Log-Likelihood:                -189.33
No. Observations:                 112   AIC:                             396.7
Df Residuals:                     103   BIC:                             421.1
Df Model:                     

## Later-period evaluation (reused)

This run selects on validation RMSE. Nevertheless, the project previously inspected the evaluation outcomes: these are exploratory retrospective results. Predictions are not rounded or clipped to improve metrics. Negative goal differences are legitimate; negative predicted goal counts are an OLS limitation.

In [7]:
table=pd.read_csv(out/'test_metrics.csv')
print(table.to_string(index=False))
print('RMSE improvement:',table.RMSE.iloc[1]-table.RMSE.iloc[0])
print('Negative predictions:',summary['negative_test_predictions'])
print(pd.read_csv(out/'test_predictions.csv').to_string(index=False))

                 model      MAE      MSE     RMSE    NRMSE        R2
             log1p_ols 0.989174 1.728010 1.314538 0.187791  0.012274
training_mean_baseline 1.066558 1.751131 1.323303 0.189043 -0.000941
RMSE improvement: 0.008765119109116482
Negative predictions: 1
match_id       date        team  actual  predicted  baseline  residual
    M083 2026-07-02    Portugal       2   1.787845  1.482143  0.212155
    M083 2026-07-02     Croatia       1   1.160072  1.482143 -0.160072
    M084 2026-07-02       Spain       3   1.601039  1.482143  1.398961
    M084 2026-07-02     Austria       0   1.112414  1.482143 -1.112414
    M085 2026-07-02 Switzerland       2   1.982464  1.482143  0.017536
    M085 2026-07-02     Algeria       0   0.936799  1.482143 -0.936799
    M086 2026-07-03   Argentina       3   2.548488  1.482143  0.451512
    M086 2026-07-03  Cape Verde       2  -0.023691  1.482143  2.023691
    M087 2026-07-03   Australia       1   1.224458  1.482143 -0.224458
    M087 2026-07-03 

![Diagnostics](results/task22/test_diagnostics.png)

![Error story](results/task22/match_error_story.png)

Task 2.2 averages paired team errors within each match. Positive bars mean regression helped relative to the baseline; this is not a significance test.

## Discussion prompts

- Is any baseline improvement practically meaningful?
- How can RMSE improve over a training-mean baseline while R² remains negative? The comparison constants differ.
- Which matches dominate errors? Map IDs through `matches_104.csv`.
- Discuss friendlies, different opponent strength, five-game variability, extra time, omitted injuries/line-ups, reused evaluation data and post-event source corrections.
- A/B order can affect the signed goal-difference analysis; paired rows and repeated teams affect independence.
- Weak prediction is a valid result, not evidence to conceal.

In [8]:
import inspect
print(inspect.getsource(run_task))

def run_task(df, features, target, task):
    out = RESULTS / task
    out.mkdir(parents=True, exist_ok=True)
    train, valid, test = [df[df['split'] == s].copy() for s in ['train', 'validation', 'test']]
    assert len(features) == 8
    assert not set(train.match_id) & set(test.match_id)
    assert not set(valid.match_id) & set(test.match_id)
    assert pd.to_datetime(train.date).max() < pd.to_datetime(valid.date).min()
    assert pd.to_datetime(valid.date).max() < pd.to_datetime(test.date).min()
    assert np.isfinite(df[features]).all().all(), 'Do not impute missing historical data'
    xt, xv, xe = [part[features].to_numpy() for part in [train, valid, test]]
    yt, yv, ye = [part[target].to_numpy() for part in [train, valid, test]]
    target_range = float(np.ptp(yt))
    assert target_range > 0
    pd.DataFrame({'feature':features, 'missing_cells':0, 'imputation_used':False}).to_csv(out/'missingness.csv', index=False)
    train[features+[target]].describe().to_csv(out/'training